# Librerias

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.2 MB/s eta 0:00:00


In [ ]:
from PIL import Image, ImageOps
import numpy as np
import tensorflow as tf
import cv2
import io
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
from skimage.morphology import skeletonize
from ultralytics import YOLO
import gc

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# ======== Custom Functions ========
def Weighted_Cross_Entropy(beta):
    def convert_to_logits(y_pred):
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        return tf.math.log(y_pred / (1 - y_pred))

    def loss(y_true, y_pred):
        y_pred = convert_to_logits(y_pred)
        loss = tf.nn.weighted_cross_entropy_with_logits(logits=y_pred, labels=y_true, pos_weight=beta)
        return tf.reduce_mean(loss)

    return loss


In [ ]:

model_segmentador = load_model('/content/model_SG.h5', custom_objects={'loss': Weighted_Cross_Entropy(10.0)}, safe_mode=False)
model_clasificador_grieta = load_model('/content/model_CG.h5', safe_mode=False)
model_detector_murosc = YOLO("/content/best.pt")
model_clasificador_ladrillo = load_model('/content/model_CL.h5', safe_mode=False)


In [ ]:
print(model_segmentador.input_shape)
print(model_clasificador_grieta.input_shape)
print(model_clasificador_ladrillo.input_shape)
print(model_detector_murosc.overrides)

(None, 512, 512, 3)
(None, 512, 512, 3)
(None, 512, 512, 3)
{'task': 'detect', 'data': '/content/Muro_Confinado-3/data.yaml', 'imgsz': 1024, 'single_cls': False, 'model': '/content/best.pt'}


In [ ]:
import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

def get_gflops(model, input_shape):
    """
    input_shape: sin batch, por ejemplo (512,512,3)
    """

    @tf.function
    def forward(x):
        return model(x)

    concrete_func = forward.get_concrete_function(
        tf.TensorSpec([1, *input_shape], tf.float32)
    )

    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()

        opts = (
            tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        )

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd='op',
            options=opts
        )

    return flops.total_float_ops / 1e9

In [ ]:
import time
import numpy as np
import tensorflow as tf

def benchmark_model(model, input_shape, warmup=30, runs=300):

    np.random.seed(42)
    tf.random.set_seed(42)

    x = tf.ones((1, *input_shape), dtype=tf.float32)

    @tf.function
    def infer(x):
        return model(x, training=False)

    # Warm-up
    for _ in range(warmup):
        y = infer(x)
        _ = tf.nest.map_structure(lambda t: t.numpy(), y)

    # Benchmark
    times = []

    for _ in range(runs):
        start = time.perf_counter()

        y = infer(x)
        _ = tf.nest.map_structure(lambda t: t.numpy(), y)

        end = time.perf_counter()
        times.append(end - start)

    times = np.array(times)

    mean_ms = times.mean() * 1000
    std_ms = times.std() * 1000
    fps = 1000 / mean_ms

    return mean_ms, std_ms, fps

In [ ]:
import time
import numpy as np
import torch
from ultralytics import YOLO


def benchmark_yolo(model,
                   input_size=(1024, 1024),
                   warmup=30,
                   runs=300,
                   device=None):

    if device is not None:
        model.to(device)

    # dummy image reproducible
    np.random.seed(42)
    x = np.random.randint(
        0, 255,
        (input_size[0], input_size[1], 3),
        dtype=np.uint8
    )

    # -------------------------
    # WARM-UP
    # -------------------------
    for _ in range(warmup):
        _ = model.predict(x, verbose=False)

    # -------------------------
    # BENCHMARK
    # -------------------------
    times = []

    for _ in range(runs):

        start = time.perf_counter()

        _ = model.predict(x, verbose=False)

        # GPU sync (important if CUDA is present)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

    times = np.array(times)

    mean_ms = times.mean() * 1000
    std_ms = times.std() * 1000
    fps = 1 / times.mean()

    print(f"Tiempo medio : {mean_ms:.2f} ms")
    print(f"Desv.Est.    : {std_ms:.2f} ms")
    print(f"FPS          : {fps:.2f}")

    return mean_ms, std_ms, fps

# Modelo Segmentador de Grietas

## GFLOPs

In [ ]:
gflops = get_gflops(model_segmentador, (512,512,3))
print(f"GFLOPs: {gflops:.3f}")

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


GFLOPs: 35.619


## FPS

In [ ]:
mean_ms, std_ms, fps = benchmark_model(
    model_segmentador,
    input_shape=(512,512,3),
    warmup=30,
    runs=300
)

print(f"Tiempo medio : {mean_ms:.2f} ms")
print(f"Desv.Est.    : {std_ms:.2f} ms")
print(f"FPS          : {fps:.2f}")

Tiempo medio : 27.47 ms
Desv.Est.    : 0.67 ms
FPS          : 36.40


# Modelo Clasifcador de Grietas

## GFLOPs

In [ ]:
gflops = get_gflops(model_clasificador_grieta, (512,512,3))
print(f"GFLOPs: {gflops:.3f}")

GFLOPs: 5.986


##FPS

In [ ]:
mean_ms, std_ms, fps = benchmark_model(
    model_clasificador_grieta,
    input_shape=(512,512,3),
    warmup=30,
    runs=300
)

print(f"Tiempo medio : {mean_ms:.2f} ms")
print(f"Desv.Est.    : {std_ms:.2f} ms")
print(f"FPS          : {fps:.2f}")

Tiempo medio : 6.57 ms
Desv.Est.    : 0.15 ms
FPS          : 152.11


# Modelos Detector de Muros

##GFLOPS

In [ ]:
model_detector_murosc.info(verbose=True, imgsz=1024)

YOLO11l summary: 358 layers, 25,311,251 parameters, 0 gradients, 223.4 GFLOPs


(358, 25311251, 0, 223.42074368)

##FPS

In [ ]:
mean_ms, std_ms, fps = benchmark_yolo(
    model_detector_murosc,
    input_size=(1024, 1024),
    warmup=30,
    runs=300
)

Tiempo medio : 72.70 ms
Desv.Est.    : 1.27 ms
FPS          : 13.76


# Modelo Clasificador de Ladrillos

##GFLOPS

In [ ]:
gflops = get_gflops(model_clasificador_ladrillo, (512,512,3))
print(f"GFLOPs: {gflops:.3f}")

GFLOPs: 5.986


##FPS

In [ ]:
mean_ms, std_ms, fps = benchmark_model(
    model_clasificador_ladrillo,
    input_shape=(512,512,3),
    warmup=30,
    runs=300
)

print(f"Tiempo medio : {mean_ms:.2f} ms")
print(f"Desv.Est.    : {std_ms:.2f} ms")
print(f"FPS          : {fps:.2f}")

Tiempo medio : 6.74 ms
Desv.Est.    : 0.28 ms
FPS          : 148.42


# Caracteristicas

In [ ]:
import platform
import psutil
import os
import tensorflow as tf
import torch

def colab_environment_report():

    print("\n======================================")
    print("   REPORTE AUTOMÁTICO DEL ENTORNO")
    print("======================================\n")

    # ================= CPU =================
    print("🧠 CPU:")
    print("Processor:", platform.processor())
    print("Architecture:", platform.machine())
    print("Physical cores:", psutil.cpu_count(logical=False))
    print("Logical cores:", psutil.cpu_count(logical=True))

    # ================= RAM =================
    ram = psutil.virtual_memory()
    print("\n🧠 RAM:")
    print(f"Total: {ram.total / (1024**3):.2f} GB")
    print(f"Available: {ram.available / (1024**3):.2f} GB")

    # ================= TENSORFLOW GPU =================
    print("\n⚡ TENSORFLOW:")
    gpus_tf = tf.config.list_physical_devices('GPU')

    if gpus_tf:
        print("GPU detected by TensorFlow:")
        for gpu in gpus_tf:
            print(" -", gpu)
    else:
        print("No GPU detected by TensorFlow (CPU mode)")

    # ================= PYTORCH / YOLO GPU =================
    print("\n🔥 PYTORCH / YOLO:")
    if torch.cuda.is_available():
        print("CUDA: YES")
        print("GPU Name:", torch.cuda.get_device_name(0))
        print("CUDA Version:", torch.version.cuda)
        print("GPU Memory (GB):",
              round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
    else:
        print("CUDA: NO (running on CPU)")

    # ================= COLAB CHECK =================
    print("\n📌 ENVIRONMENT:")

    if 'COLAB_GPU' in os.environ:
        print("Running on Google Colab")
        print("GPU runtime detected: YES")
    else:
        print("Running locally or CPU runtime")

    # ================= FINAL INTERPRETATION =================
    print("\n🧾 SUMMARY:")

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"Active hardware: GPU ({gpu_name}) + CPU support")
    else:
        print("Active hardware: CPU only")

    print("\n======================================\n")

# ejecutar
colab_environment_report()


   REPORTE AUTOMÁTICO DEL ENTORNO

🧠 CPU:
Processor: x86_64
Architecture: x86_64
Physical cores: 1
Logical cores: 2

🧠 RAM:
Total: 12.67 GB
Available: 9.24 GB

⚡ TENSORFLOW:
GPU detected by TensorFlow:
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

🔥 PYTORCH / YOLO:
CUDA: YES
GPU Name: Tesla T4
CUDA Version: 12.8
GPU Memory (GB): 14.56

📌 ENVIRONMENT:
Running on Google Colab
GPU runtime detected: YES

🧾 SUMMARY:
Active hardware: GPU (Tesla T4) + CPU support


